---
# 📂 Notebook: `limpieza_olist.ipynb`
---

In [ ]:
import os
import warnings
import unicodedata

import numpy as np
import pandas as pd

# Silenciar solo ruido conocido: cambios de API en pandas y categorias deprecadas.
# Otros warnings (p.ej. SettingWithCopy real) seguiran visibles.
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Rutas de la primera etapa del pipeline.
# Entrada: CSV originales de Olist.
# Salida: CSV limpios con los nombres que consume el script 02.
DATA_PATH = os.environ.get('DATA_PATH', '../csv/csv_originales/')
OUTPUT_PATH = os.environ.get('OUTPUT_PATH', '../csv/csv_limpios/')

DATA_PATH = os.path.normpath(DATA_PATH)
OUTPUT_PATH = os.path.normpath(OUTPUT_PATH)
os.makedirs(OUTPUT_PATH, exist_ok=True)

PIPELINE_AUDIT = {
    'leidos': {},
    'generados': {},
}


def input_csv_path(filename):
    return os.path.join(DATA_PATH, filename)


def output_csv_path(filename):
    return os.path.join(OUTPUT_PATH, filename)


def registrar_lectura(nombre, df):
    PIPELINE_AUDIT['leidos'][nombre] = {
        'ruta': input_csv_path(nombre),
        'filas': len(df),
        'columnas': list(df.columns),
    }


def registrar_generado(nombre, df, filas_antes):
    PIPELINE_AUDIT['generados'][nombre] = {
        'ruta': output_csv_path(nombre),
        'filas_antes': filas_antes,
        'filas_despues': len(df),
        'columnas_finales': list(df.columns),
        'nulos_principales': df.isna().sum().sort_values(ascending=False).head(10).to_dict(),
    }


def leer_csv_original(nombre, **kwargs):
    ruta = input_csv_path(nombre)
    df = pd.read_csv(ruta, **kwargs)
    registrar_lectura(nombre, df)
    return df


def guardar_csv_limpio(df, nombre, filas_antes, **kwargs):
    ruta = output_csv_path(nombre)
    df.to_csv(ruta, index=False, **kwargs)
    registrar_generado(nombre, df, filas_antes)
    return ruta

print(f'Entrada configurada: {DATA_PATH}')
print(f'Salida configurada : {OUTPUT_PATH}')

---
## 1. `olist_customers_dataset`

In [ ]:
customers = leer_csv_original('olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})

print(f'Filas : {customers.shape[0]:,}  |  Columnas : {customers.shape[1]}')
print(f'\nTipos de datos:\n{customers.dtypes}')
print(f'\nNulos por columna:\n{customers.isnull().sum()}')
print(f'\nDuplicados (fila completa): {customers.duplicated().sum():,}')
print(f'Duplicados en customer_id:  {customers["customer_id"].duplicated().sum():,}')
customers.head()

In [ ]:
customers_clean = customers.copy()

customers_clean['customer_zip_code_prefix'] = (customers_clean['customer_zip_code_prefix'].str.strip().str.zfill(5))

customers_clean['customer_city'] = (customers_clean['customer_city'].str.strip().str.title())

print(customers_clean[['customer_zip_code_prefix', 'customer_city']].head(5).to_string())
print(f'\nFilas antes:  {customers.shape[0]:,}')
print(f'Filas después: {customers_clean.shape[0]:,}')
if customers.shape[0] == customers_clean.shape[0]:
    print('No se eliminaron filas')
else:
    print(f'Se eliminaron {customers.shape[0]-customers_clean.shape[0]:,} filas')

In [ ]:
guardar_csv_limpio(customers_clean, 'olist_customers_clean.csv', len(customers))
customers_clean.head()

---
## 2. `olist_order_items_dataset`

In [ ]:
order_items = leer_csv_original('olist_order_items_dataset.csv')

print(f'Filas: {order_items.shape[0]:,}  |  Columnas: {order_items.shape[1]}')
print(f'\nTipos de datos:\n{order_items.dtypes}')
print(f'\nNulos por columna:\n{order_items.isnull().sum()}')
print(f'\nDuplicados (fila completa): {order_items.duplicated().sum():,}')
print(f'\nEstadísticas de precio y flete:')
print(order_items[['price', 'freight_value']].describe())
print(f'\nRegistros con flete = 0: {(order_items["freight_value"] == 0).sum():,}')
print(f'Registros con precio = 0: {(order_items["price"] == 0).sum():,}')
order_items.head()

In [ ]:
order_items_clean = order_items.copy()

order_items_clean['shipping_limit_date'] = pd.to_datetime(order_items_clean['shipping_limit_date'],errors='coerce')

nat_count = order_items_clean['shipping_limit_date'].isna().sum()
print(f'Fechas que no pudieron convertirse (NaT): {nat_count}')

In [ ]:
guardar_csv_limpio(order_items_clean, 'olist_order_items_clean.csv', len(order_items))
order_items_clean.head()

---
## 3. `olist_geolocation_dataset`

In [ ]:
geo = leer_csv_original('olist_geolocation_dataset.csv', dtype={'geolocation_zip_code_prefix': str})

print(f'Filas: {geo.shape[0]:,}  |  Columnas: {geo.shape[1]}')
print(f'\nTipos de datos:\n{geo.dtypes}')
print(f'\nNulos por columna:\n{geo.isnull().sum()}')
print(f'\nDuplicados (fila completa): {geo.duplicated().sum():,}')
print(f'Zips únicos: {geo["geolocation_zip_code_prefix"].nunique():,}')
print(f'\nEstadísticas de coordenadas:')
print(geo[['geolocation_lat', 'geolocation_lng']].describe())
geo.head()

In [ ]:
geo_clean = geo.drop_duplicates()
print(f'Filas antes:  {geo.shape[0]:,}')
print(f'Filas después: {geo_clean.shape[0]:,}')
print(f'Eliminadas:   {geo.shape[0] - geo_clean.shape[0]:,}')

In [ ]:
# Límites geográficos de Brasil (chance estén mal)
LAT_MIN, LAT_MAX = -33.75, 5.27
LNG_MIN, LNG_MAX = -73.98, -34.79

fuera_rango = geo_clean[(geo_clean['geolocation_lat'] < LAT_MIN)|(geo_clean['geolocation_lat'] > LAT_MAX)|(geo_clean['geolocation_lng'] < LNG_MIN)|(geo_clean['geolocation_lng'] > LNG_MAX)]
print(f'Registros con coordenadas fuera de Brasil: {len(fuera_rango):,}')
print(fuera_rango[['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng','geolocation_city']].head(10))

geo_clean = geo_clean[
    (geo_clean['geolocation_lat'] >= LAT_MIN) &
    (geo_clean['geolocation_lat'] <= LAT_MAX) &
    (geo_clean['geolocation_lng'] >= LNG_MIN) &
    (geo_clean['geolocation_lng'] <= LNG_MAX)
]
print(f'\nFilas después de filtrar coordenadas: {geo_clean.shape[0]:,}')

In [ ]:
def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    texto_normalizado = unicodedata.normalize('NFKD', str(texto))
    texto_sin_acentos = ''.join(
        c for c in texto_normalizado
        if not unicodedata.combining(c)
    )
    return texto_sin_acentos.lower().strip()

# Muestra antes
print('Muestra ANTES:')
print(geo_clean['geolocation_city'].unique()[:8])

# Aplicar normalización
geo_clean = geo_clean.copy()
geo_clean['geolocation_city'] = geo_clean['geolocation_city'].apply(normalizar_texto)

# Muestra después
print('\nMuestra DESPUÉS:')
print(geo_clean['geolocation_city'].unique()[:8])

# Verificar que 'são paulo' y 'sao paulo' ahora son iguales
print(f'\nCiudades únicas antes de normalizar: (ya aplicado en paso anterior)')
print(f'Ciudades únicas después de normalizar: {geo_clean["geolocation_city"].nunique():,}')

In [ ]:
geo_clean['geolocation_zip_code_prefix'] = (geo_clean['geolocation_zip_code_prefix'].str.strip().str.zfill(5))

geo_agg = geo_clean.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat   = ('geolocation_lat',  'mean'),
    geolocation_lng   = ('geolocation_lng',  'mean'),
    geolocation_city  = ('geolocation_city',  lambda x: x.mode()[0]),
    geolocation_state = ('geolocation_state', lambda x: x.mode()[0])
).reset_index()

geo_agg['geolocation_city'] = geo_agg['geolocation_city'].str.title()

print(f'Zips únicos en el resultado: {geo_agg.shape[0]:,}')
print(f'\nResumen del proceso:')
print(f'  Filas originales:          {geo.shape[0]:,}')
print(f'  Tras eliminar duplicados:  {geo.shape[0] - 261831:,}')
print(f'  Tras filtrar coordenadas:  {geo_clean.shape[0]:,}')
print(f'  Tras agregar por zip:      {geo_agg.shape[0]:,}  ← tabla final')
geo_agg.head()

In [ ]:
guardar_csv_limpio(geo_agg, 'olist_geolocation_clean.csv', len(geo))
geo_agg.head()

---
# 📂 Notebook: `limpieza_orders_products.ipynb`
---

# Documentación del Proceso de Limpieza: Products y Orders

## 1. Limpieza del Dataset de Productos (`olist_products_dataset.csv`)
El objetivo principal en esta tabla fue preservar todos los registros (IDs de productos) para no perder transacciones en el modelo OLAP final. Se optó por la imputación de valores nulos en lugar de la eliminación de filas.

* **Normalización de Categorías (`product_category_name`):**
  * Se identificaron 610 valores nulos y se rellenaron con la etiqueta genérica `'Sin Categoria'`.
  * Se aplicó limpieza de formato de texto: se reemplazaron los guiones bajos (`_`) por espacios y se aplicó formato de título (Title Case) para estandarizar la visualización en los Dashboards (ej. pasó de `esporte_lazer` a `Esporte Lazer`).
* **Imputación de Variables Numéricas:**
  * Las columnas correspondientes a dimensiones, peso y cantidad de fotografías contenían valores nulos esporádicos.
  * Estos vacíos se imputaron utilizando la **mediana** de cada respectiva columna. Se eligió la mediana por encima de la media geométrica o el promedio para evitar que la asimetría de valores atípicos (outliers) distorsionara los datos.

## 2. Limpieza del Dataset de Pedidos (`olist_orders_dataset.csv`)
El enfoque para la tabla de pedidos transaccionales fue estandarizar los datos categóricos y asegurar la integridad de los tipos de datos temporales (Time Series) para el posterior análisis logístico.

* **Estandarización de Estatus (`order_status`):**
  * Se eliminaron espacios en blanco accidentales (`strip`) y se capitalizó el texto para mantener consistencia en agrupaciones futuras (ej. `Delivered`, `Canceled`).
* **Conversión de Fechas (Formato Datetime):**
  * Las 5 columnas relacionadas con marcas de tiempo (compra, aprobación, envío a paquetería, entrega y fecha estimada) se convirtieron rigurosamente de texto (String) a formato `datetime64`.
  * **Decisión de Negocio sobre Nulos:** Se conservaron intencionalmente los valores nulos generados en las columnas de entrega (`order_delivered_customer_date`, etc.). Esto respeta la naturaleza operativa del negocio, ya que un pedido con estatus `Canceled` o `Invoiced` lógicamente carece de una marca de tiempo de entrega final.
* **Ingeniería de Características (Feature Engineering):**
  * Se extrajo y generó una nueva columna llamada **`fecha_compra_dia`** a partir del timestamp original, aislando únicamente la fecha (YYYY-MM-DD).
  * *Propósito:* Esta característica fue construida estratégicamente para servir como llave foránea (JOIN key) al momento de integrar **fuentes exógenas**, permitiendo cruzar cada pedido con el clima exacto de ese día.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import os
import requests

warnings.filterwarnings('ignore')


# ==========================================
# 0. CONFIGURACIÓN INICIAL
# ==========================================

In [ ]:
# DATA_PATH ya esta definido mas arriba via env var (default '../csv/csv_originales/')
print("Cargando archivos...")

# Leemos cada archivo CSV
products    = leer_csv_original('olist_products_dataset.csv')
orders      = leer_csv_original('olist_orders_dataset.csv')

print("¡Todos los archivos se cargaron con éxito!")

# ==========================================
# 1. LIMPIEZA DE PRODUCTS
# ==========================================

In [ ]:
print("Limpiando Products...")
products_clean = products.copy()

In [ ]:
# Rellenar categorías vacías y darle formato (sin guiones bajos y con Mayúsculas)
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('sin_categoria')
products_clean['product_category_name'] = products_clean['product_category_name'].str.replace('_', ' ').str.title()

In [ ]:
# Rellenar dimensiones/pesos vacíos con la mediana para no perder el registro
cols_numericas = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                  'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in cols_numericas:
    products_clean[col] = products_clean[col].fillna(products_clean[col].median())

print(f"✅ Products limpio (Filas: {products_clean.shape[0]:,})")

# ==========================================
# 2. LIMPIEZA DE ORDERS
# ==========================================

In [ ]:
print("\nLimpiando Orders...")
orders_clean = orders.copy()

In [ ]:
# Normalizar el texto del estatus
orders_clean['order_status'] = orders_clean['order_status'].str.strip().str.title()

In [ ]:
# Convertir textos a formato Fecha (Datetime)
date_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in date_columns:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')

In [ ]:
# (Opcional pero recomendado) Extraemos solo el día de la compra en una columna nueva
# Esto te servirá muchísimo cuando decidas conectar el clima más adelante
orders_clean['fecha_compra_dia'] = orders_clean['order_purchase_timestamp'].dt.date

print(f"✅ Orders limpio (Filas: {orders_clean.shape[0]:,})")

# ==========================================
# 3. EXPORTAR ARCHIVOS LIMPIOS
# ==========================================

In [ ]:
import os

print("\nExportando archivos...")

# OUTPUT_PATH ya esta definido mas arriba via env var (default '../csv/csv_limpios/')
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Guardamos los archivos
guardar_csv_limpio(products_clean, 'olist_products_clean.csv', len(products))
guardar_csv_limpio(orders_clean, 'olist_orders_clean.csv', len(orders))

print(f"¡Listo! Archivos guardados en la carpeta: {OUTPUT_PATH}")

---
# 📂 Notebook: `limpieza_payments.ipynb`
---

# Limpieza de olist_order_payments_dataset.csv

Este notebook carga, limpia y valida el dataset de pagos de Olist.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
# Carga del dataset
file_path = input_csv_path('olist_order_payments_dataset.csv')
df = leer_csv_original('olist_order_payments_dataset.csv')

print(f'Filas iniciales: {len(df):,}')
display(df.head())
display(df.info())

In [ ]:
# Diagnostico inicial
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)

print('Duplicados exactos:', int(df.duplicated().sum()))
display(missing)
display(df.describe(include='all').T)

In [ ]:
# Limpieza
df_clean = df.copy()

# 1) Estandarizar nombres de columnas
df_clean.columns = [c.strip().lower() for c in df_clean.columns]

# 2) Limpiar campos de texto
df_clean['order_id'] = df_clean['order_id'].astype('string').str.strip().str.lower()
df_clean['payment_type'] = df_clean['payment_type'].astype('string').str.strip().str.lower()

# 3) Convertir tipos numericos
num_cols = ['payment_sequential', 'payment_installments', 'payment_value']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 4) Eliminar duplicados exactos
before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates()
removed_dup = before_dup - len(df_clean)

# 5) Mantener la mayor cantidad de datos: solo eliminar filas sin order_id usable
missing_order_id = df_clean['order_id'].isna() | df_clean['order_id'].isin(['', 'nan'])
removed_missing_order_id = int(missing_order_id.sum())
df_clean = df_clean[~missing_order_id].copy()

# 6) Imputacion conservadora para no perder registros
na_seq = int(df_clean['payment_sequential'].isna().sum())
na_inst = int(df_clean['payment_installments'].isna().sum())
na_val = int(df_clean['payment_value'].isna().sum())
df_clean['payment_sequential'] = df_clean['payment_sequential'].fillna(1)
df_clean['payment_installments'] = df_clean['payment_installments'].fillna(1)
df_clean['payment_value'] = df_clean['payment_value'].fillna(0)

# 7) Eliminar solo valores imposibles (negativos)
before_range = len(df_clean)
df_clean = df_clean[(df_clean['payment_sequential'] >= 1) &
                    (df_clean['payment_installments'] >= 0) &
                    (df_clean['payment_value'] >= 0)].copy()
removed_invalid_values = before_range - len(df_clean)

# 8) Tipos finales
df_clean['payment_sequential'] = df_clean['payment_sequential'].astype('int64')
df_clean['payment_installments'] = df_clean['payment_installments'].astype('int64')
df_clean['payment_value'] = df_clean['payment_value'].astype('float64')
df_clean['payment_type'] = df_clean['payment_type'].replace({'': np.nan, 'nan': np.nan}).fillna('unknown')

print(f'Duplicados eliminados: {removed_dup}')
print(f'Filas sin order_id eliminadas: {removed_missing_order_id}')
print(f'Nulos imputados en payment_sequential: {na_seq}')
print(f'Nulos imputados en payment_installments: {na_inst}')
print(f'Nulos imputados en payment_value: {na_val}')
print(f'Filas eliminadas por valores imposibles: {removed_invalid_values}')
print(f'Filas finales: {len(df_clean):,}')

In [ ]:
# Revision final de calidad
display(df_clean.head())
display(df_clean.isna().sum().to_frame('missing_count'))
display(df_clean.describe(include='all').T)

print('Duplicados exactos finales:', int(df_clean.duplicated().sum()))

In [ ]:
# Exportar dataset limpio
output_path = guardar_csv_limpio(df_clean, 'olist_order_payments_dataset_clean.csv', len(df))
print(f'Archivo guardado en: {output_path}')

In [ ]:
# Carga del dataset
file_path = output_csv_path('olist_order_payments_dataset_clean.csv')
df = pd.read_csv(file_path)

print(f'Filas iniciales: {len(df):,}')
display(df.head())
display(df.info())

---
# 📂 Notebook: `limpieza_product_category.ipynb`
---

# Limpieza de `product_category_name_translation.csv`

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Carga de datos
_FILENAME = 'product_category_name_translation.csv'
_data_path = input_csv_path(_FILENAME)
if not os.path.exists(_data_path):
    raise FileNotFoundError(
        f"No se encontro '{_FILENAME}' en la ruta de entrada configurada: {_data_path}"
    )

print(f"Cargando datos desde: {os.path.abspath(_data_path)}")
df = pd.read_csv(_data_path, encoding='utf-8-sig')
registrar_lectura(_FILENAME, df)
df.head()

## 1. Exploración inicial

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nColumnas:", df.columns.tolist())
print("\n=== Valores nulos por columna ===")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("Duplicados en product_category_name:", df['product_category_name'].duplicated().sum())
print("Duplicados en product_category_name_english:", df['product_category_name_english'].duplicated().sum())

In [ ]:
df

## 2. Detección de erratas en `product_category_name_english`

La columna en portugués está limpia (snake_case, sin caracteres especiales). Sin embargo, la columna en inglés contiene erratas ortográficas que se detectan a continuación.

In [ ]:
ERRATAS = {
    'costruction':  'construction',
    'fashio_':      'fashion_',
    'confort':      'comfort',
    'craftmanship': 'craftsmanship',
}

print("Erratas detectadas en product_category_name_english:\n")
en = df['product_category_name_english']
for typo, fix in ERRATAS.items():
    matches = en[en.str.contains(typo, na=False)]
    if not matches.empty:
        for val in matches:
            corrected = val.replace(typo, fix)
            print(f"  {val!r:40s}  ->  {corrected!r}")
print(f"\nTotal de valores afectados: {sum(en.str.contains(t, na=False).sum() for t in ERRATAS)}")

## 3. Corrección de erratas

In [ ]:
df['english_original'] = df['product_category_name_english']

for typo, fix in ERRATAS.items():
    df['product_category_name_english'] = (
        df['product_category_name_english'].str.replace(typo, fix, regex=False)
    )

# Mostrar cambios realizados
mascara = df['product_category_name_english'] != df['english_original']
cambios = df[mascara][['english_original', 'product_category_name_english']].copy()
cambios.columns = ['original', 'corregido']
print(f"Erratas corregidas: {len(cambios)}")
cambios.reset_index(drop=True)

## 4. Reporte final

In [ ]:
df_clean = df.drop(columns=['english_original'])

print("=" * 50)
print("REPORTE DE LIMPIEZA")
print("=" * 50)
print(f"\nFilas totales       : {len(df_clean)}")
print(f"Columnas            : {df_clean.columns.tolist()}")
print(f"\nValores nulos:")
print('Nulos por columna en df_clean:')
print(df_clean.isnull().sum().to_string())
print(f"\nDuplicados          : {df_clean.duplicated().sum()}")
print(f"Erratas corregidas  : {mascara.sum()}")

# Verificar que ya no hay erratas
residual = 0
for typo in ERRATAS:
    residual += df_clean['product_category_name_english'].str.contains(typo, na=False).sum()
print(f"Erratas residuales  : {residual}")

## 5. Exportar dataset limpio

In [ ]:
# Guardar sin BOM (encoding utf-8 estandar)
output_path = guardar_csv_limpio(
    df_clean,
    'product_category_name_translation_clean.csv',
    PIPELINE_AUDIT['leidos']['product_category_name_translation.csv']['filas'],
    encoding='utf-8'
)
print(f"Dataset limpio guardado en: {os.path.abspath(output_path)}")
print(f"Forma final: {df_clean.shape}")

---
# 📂 Notebook: `limpieza_reviews.ipynb`
---

# Limpieza de olist_order_reviews_dataset.csv

Flujo conservador para limpiar reviews sin eliminar datos validos de mas.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

In [ ]:
# Carga del dataset
file_path = input_csv_path('olist_order_reviews_dataset.csv')
df = leer_csv_original('olist_order_reviews_dataset.csv')

print(f'Filas iniciales: {len(df):,}')
display(df.head())
display(df.info())

In [ ]:
# Diagnostico de precision (para decidir limpieza sin perder datos validos)
n = len(df)
df_diag = df.copy()

# 1) Nulos por columna con clasificacion de criticidad
missing = df_diag.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / n * 100).round(2)
missing['criticidad'] = 'alta'
missing.loc[['review_comment_title', 'review_comment_message'], 'criticidad'] = 'baja (campo opcional)'

# 2) Normalizacion minima de llaves solo para diagnosticar
for col in ['review_id', 'order_id']:
    df_diag[col] = df_diag[col].astype('string').str.strip().str.lower()

missing_review_id = int((df_diag['review_id'].isna() | df_diag['review_id'].isin(['', 'nan'])).sum())
missing_order_id = int((df_diag['order_id'].isna() | df_diag['order_id'].isin(['', 'nan'])).sum())
dup_exact = int(df_diag.duplicated().sum())
dup_review_id = int(df_diag.duplicated(subset=['review_id']).sum())
dup_pair = int(df_diag.duplicated(subset=['review_id', 'order_id']).sum())

# 3) Consistencia de review_score
score_num = pd.to_numeric(df_diag['review_score'], errors='coerce')
invalid_score_type = int(score_num.isna().sum())
score_out_of_range = int(((score_num < 1) | (score_num > 5)).fillna(False).sum())
score_non_integer = int(((score_num % 1 != 0) & score_num.notna()).sum())

# 4) Calidad temporal
creation_dt = pd.to_datetime(df_diag['review_creation_date'], errors='coerce')
answer_dt = pd.to_datetime(df_diag['review_answer_timestamp'], errors='coerce')
invalid_creation_date = int(creation_dt.isna().sum())
invalid_answer_date = int(answer_dt.isna().sum())
answer_before_creation = int(((answer_dt < creation_dt) & answer_dt.notna() & creation_dt.notna()).sum())

# 5) Texto libre: faltantes esperados y placeholders
for col in ['review_comment_title', 'review_comment_message']:
    df_diag[col] = df_diag[col].astype('string').str.replace(r'\\s+', ' ', regex=True).str.strip()
placeholder_pattern = r'^(na|n/a|null|none|-)$'
placeholder_title = int(df_diag['review_comment_title'].str.fullmatch(placeholder_pattern, case=False, na=False).sum())
placeholder_message = int(df_diag['review_comment_message'].str.fullmatch(placeholder_pattern, case=False, na=False).sum())
missing_title = int(df_diag['review_comment_title'].isna().sum())
missing_message = int(df_diag['review_comment_message'].isna().sum())
short_message = int((df_diag['review_comment_message'].str.len().fillna(0).between(1, 3)).sum())

# 6) Casos potencialmente ambiguos en duplicados por review_id
review_groups = df_diag.groupby('review_id', dropna=False).agg(
    n_orders=('order_id', 'nunique'),
    n_scores=('review_score', 'nunique')
)
multi_order_review = int((review_groups['n_orders'] > 1).sum())
multi_score_review = int((review_groups['n_scores'] > 1).sum())

# 7) Matriz de decisiones sugeridas (accion por regla)
rules = pd.DataFrame([
    {'criterio': 'Fila sin review_id', 'afectadas': missing_review_id, 'accion_sugerida': 'eliminar'},
    {'criterio': 'Fila sin order_id', 'afectadas': missing_order_id, 'accion_sugerida': 'eliminar'},
    {'criterio': 'Duplicado exacto', 'afectadas': dup_exact, 'accion_sugerida': 'eliminar'},
    {'criterio': 'Duplicado por review_id', 'afectadas': dup_review_id, 'accion_sugerida': 'marcar bandera (no eliminar directo)'},
    {'criterio': 'Duplicado por (review_id, order_id)', 'afectadas': dup_pair, 'accion_sugerida': 'revisar y posible deduplicacion puntual'},
    {'criterio': 'review_score no numerico', 'afectadas': invalid_score_type, 'accion_sugerida': 'imputar conservador'},
    {'criterio': 'review_score fuera de [1,5]', 'afectadas': score_out_of_range, 'accion_sugerida': 'clip al rango'},
    {'criterio': 'review_score no entero', 'afectadas': score_non_integer, 'accion_sugerida': 'round'},
    {'criterio': 'review_answer < review_creation', 'afectadas': answer_before_creation, 'accion_sugerida': 'marcar bandera temporal'},
    {'criterio': 'review_comment_title nulo', 'afectadas': missing_title, 'accion_sugerida': 'conservar (campo opcional)'},
    {'criterio': 'review_comment_message nulo', 'afectadas': missing_message, 'accion_sugerida': 'conservar (campo opcional)'},
    {'criterio': 'Placeholder en title', 'afectadas': placeholder_title, 'accion_sugerida': 'convertir a NA'},
    {'criterio': 'Placeholder en message', 'afectadas': placeholder_message, 'accion_sugerida': 'convertir a NA'},
    {'criterio': 'Mensaje muy corto (1-3 chars)', 'afectadas': short_message, 'accion_sugerida': 'mantener y marcar para analisis'}
])
rules['pct'] = (rules['afectadas'] / n * 100).round(2)

print(f'Filas totales: {n:,}')
print('Duplicados exactos:', dup_exact)
print('Duplicados por review_id:', dup_review_id)
print('Duplicados por (review_id, order_id):', dup_pair)
print('Review_id en multiples ordenes:', multi_order_review)
print('Review_id con multiples scores:', multi_score_review)

display(missing)
display(df_diag['review_score'].value_counts(dropna=False).sort_index().to_frame('count'))
display(rules.sort_values(by=['afectadas', 'criterio'], ascending=[False, True]).reset_index(drop=True))
display(df_diag.describe(include='all').T)

In [ ]:
# Limpieza conservadora
df_clean = df.copy()

# 1) Estandarizar nombres de columnas
df_clean.columns = [c.strip().lower() for c in df_clean.columns]

# 2) Limpiar campos de texto de identificadores
id_cols = ['review_id', 'order_id']
for col in id_cols:
    df_clean[col] = df_clean[col].astype('string').str.strip()

# 3) Limpiar texto libre sin borrar informacion
text_cols = ['review_comment_title', 'review_comment_message']
for col in text_cols:
    df_clean[col] = df_clean[col].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    df_clean[col] = df_clean[col].replace({'': pd.NA, 'nan': pd.NA})

# 3b) Convertir placeholders a NA (detectados en diagnostico)
placeholder_pattern = r'^(?:no\s*comment|n/?a|\.+|-+|\?+|test|xxx)$'
for col in text_cols:
    mask_placeholder = df_clean[col].str.match(placeholder_pattern, case=False, na=False)
    print(f'Placeholders detectados en {col}: {int(mask_placeholder.sum())}')
    df_clean.loc[mask_placeholder, col] = pd.NA

missing_title_after_clean = int(df_clean['review_comment_title'].isna().sum())
missing_message_after_clean = int(df_clean['review_comment_message'].isna().sum())

# 3c) Bandera para mensajes muy cortos (1-3 chars) — mantener y marcar
df_clean['flag_short_message'] = (
    df_clean['review_comment_message'].notna()
    & (df_clean['review_comment_message'].str.len() <= 3)
)
print(f'Mensajes muy cortos (1-3 chars): {int(df_clean["flag_short_message"].sum())}')

# 4) Convertir review_score a numerico e imputar de forma conservadora
df_clean['review_score'] = pd.to_numeric(df_clean['review_score'], errors='coerce')
na_score = int(df_clean['review_score'].isna().sum())
if na_score > 0:
    median_score = int(round(df_clean['review_score'].median(skipna=True)))
    median_score = min(5, max(1, median_score))
else:
    median_score = 5
df_clean['review_score'] = df_clean['review_score'].fillna(median_score)

# 5) Ajustar review_score al rango valido [1, 5]
out_of_range_before = int(((df_clean['review_score'] < 1) | (df_clean['review_score'] > 5)).sum())
df_clean['review_score'] = df_clean['review_score'].clip(1, 5).round().astype('int64')

# 6) Eliminar solo registros sin llaves minimas
missing_review_id = df_clean['review_id'].isna() | df_clean['review_id'].isin(['', 'nan'])
missing_order_id = df_clean['order_id'].isna() | df_clean['order_id'].isin(['', 'nan'])
rows_without_keys = int((missing_review_id | missing_order_id).sum())
df_clean = df_clean[~(missing_review_id | missing_order_id)].copy()

# 7) Fechas: parsear y conservar nulos si existen
df_clean['review_creation_date'] = pd.to_datetime(df_clean['review_creation_date'], errors='coerce')
df_clean['review_answer_timestamp'] = pd.to_datetime(df_clean['review_answer_timestamp'], errors='coerce')
invalid_creation_date = int(df_clean['review_creation_date'].isna().sum())
invalid_answer_date = int(df_clean['review_answer_timestamp'].isna().sum())

# 8) Bandera de calidad temporal (sin eliminar)
df_clean['flag_answer_before_creation'] = (
    df_clean['review_answer_timestamp'].notna()
    & df_clean['review_creation_date'].notna()
    & (df_clean['review_answer_timestamp'] < df_clean['review_creation_date'])
)

# 9) Banderas para review_id repetido (no se elimina)
df_clean['flag_review_id_duplicated'] = df_clean.duplicated(subset=['review_id'], keep=False)
df_clean['flag_review_id_multi_order'] = df_clean.groupby('review_id')['order_id'].transform('nunique').gt(1)

# 10) Eliminar duplicados exactos al final (sin contar columnas de banderas)
flag_cols = [c for c in df_clean.columns if c.startswith('flag_')]
original_cols = [c for c in df_clean.columns if c not in flag_cols]
before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=original_cols)
removed_dup = before_dup - len(df_clean)

# --- Reporte final ---
print(f'\n{"="*50}')
print(f'REPORTE DE LIMPIEZA')
print(f'{"="*50}')
print(f'Filas sin llaves minimas eliminadas: {rows_without_keys}')
print(f'Duplicados exactos eliminados: {removed_dup}')
print(f'Review_id repetidos (conservados): {int(df_clean["flag_review_id_duplicated"].sum())}')
print(f'Review_id en multiples ordenes: {int(df_clean["flag_review_id_multi_order"].sum())}')
print(f'Nulos imputados en review_score: {na_score}')
print(f'Review_score fuera de rango (clip): {out_of_range_before}')
print(f'Comentarios sin titulo (esperado): {missing_title_after_clean}')
print(f'Comentarios sin mensaje (esperado): {missing_message_after_clean}')
print(f'Mensajes cortos (flag): {int(df_clean["flag_short_message"].sum())}')
print(f'Fechas creacion no parseables: {invalid_creation_date}')
print(f'Fechas respuesta no parseables: {invalid_answer_date}')
print(f'Filas con answer < creation: {int(df_clean["flag_answer_before_creation"].sum())}')
print(f'Filas finales: {len(df_clean):,}')

In [ ]:
# Revision final de calidad
display(df_clean.head())
display(df_clean.isna().sum().to_frame('missing_count'))
display(df_clean.describe(include='all').T)

print('Duplicados exactos finales:', int(df_clean.duplicated().sum()))
print('Duplicados por review_id finales:', int(df_clean.duplicated(subset=['review_id']).sum()))
print('Filas marcadas con review_id repetido:', int(df_clean['flag_review_id_duplicated'].sum()))
print('Filas marcadas con review_id en multiples ordenes:', int(df_clean['flag_review_id_multi_order'].sum()))

# Revisar qué convirtió a NA el patrón de placeholders
mask = df_clean['review_comment_message'].isna() & df['review_comment_message'].notna()
print(df.loc[mask, 'review_comment_message'].value_counts().head(20))

In [ ]:
# Exportar dataset limpio
output_path = guardar_csv_limpio(df_clean, 'olist_order_reviews_dataset_clean.csv', len(df))
print(f'Archivo guardado en: {output_path}')

---
# 📂 Notebook: `limpieza_sellers_data.ipynb`
---

# Limpieza de `olist_seller_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Carga de datos
# dtype str en seller_zip_code_prefix para preservar ceros iniciales
_FILENAME = 'olist_sellers_dataset.csv'
_data_path = input_csv_path(_FILENAME)
if not os.path.exists(_data_path):
    raise FileNotFoundError(
        f"No se encontro '{_FILENAME}' en la ruta de entrada configurada: {_data_path}"
    )

print(f"Cargando datos desde: {os.path.abspath(_data_path)}")
df = pd.read_csv(_data_path, dtype={'seller_zip_code_prefix': str})
registrar_lectura(_FILENAME, df)
df.head()

## 1. Exploración inicial

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nPrimeras filas:")
df.head()


In [ ]:
print("=== Valores nulos por columna ===")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("IDs de vendedor duplicados:", df['seller_id'].duplicated().sum())
print("\nEstados únicos:", sorted(df['seller_state'].unique().tolist()))
print("\nCiudades únicas:", df['seller_city'].nunique())


## 2. Análisis de `seller_city`

La columna `seller_city` presenta varios tipos de inconsistencias que se detallan a continuación.

In [ ]:
import re
import unicodedata

cities = df['seller_city']

problemas = {
    'Con barra (ciudad/estado o ciudad/ciudad)': cities[cities.str.contains(r'[/\\]', regex=True, na=False)].unique().tolist(),
    'Con coma (ciudad, estado, país)':           cities[cities.str.contains(r',', na=False)].unique().tolist(),
    'Con paréntesis':                            cities[cities.str.contains(r'\(', na=False)].unique().tolist(),
    'Con acentos':                               cities[cities.str.contains(r'[àáâãäéêëíïóôõöúüçñ]', regex=True, na=False)].unique().tolist(),
    'Con apóstrofe especial (´)':               cities[cities.str.contains('´', na=False)].unique().tolist(),
    'Solo dígitos':                              cities[cities.str.fullmatch(r'\d+', na=False)].unique().tolist(),
    'Email':                                     cities[cities.str.contains('@', na=False)].unique().tolist(),
    'Abreviatura de estado (sp, rj…)':          cities[cities.isin(['sp', 'rj', 'es', 'pr', 'mg'])].unique().tolist(),
    'Estado incluido en texto (ej. brasilia df)': cities[cities.str.contains(r'\bdf\b|\bmg\b|\brs\b', regex=True, na=False)
                                                         & ~cities.str.contains(r'[/\\,]', na=False)].unique().tolist(),
}

for tipo, vals in problemas.items():
    if vals:
        print(f"\n[{tipo}] ({len(vals)} valores únicos):")
        for v in vals:
            print(f"    {v!r}")


## 3. Limpieza de `seller_city`

Pasos aplicados en orden:
1. Valores irrecuperables (solo dígitos, email) → `NaN`
2. Eliminar acentos para coherencia con el resto del dataset
3. Normalizar apóstrofes especiales (`´` → `'`)
4. Extraer parte antes de `/` o `\` (formatos `ciudad/estado`)
5. Extraer parte antes de la primera coma (formatos `ciudad, estado, país`)
6. Eliminar contenido entre paréntesis
7. Correcciones puntuales: abreviaturas (`sp` → `sao paulo`) y sufijos de estado (`brasilia df` → `brasilia`)

In [ ]:
REPLACEMENTS = {
    'sp':          'sao paulo',
    'brasilia df': 'brasilia',
}

def clean_city(city: str) -> str:
    if not isinstance(city, str):
        return np.nan

    city = city.strip().lower()

    # 1. Valores irrecuperables: solo dígitos o dirección de email
    if re.fullmatch(r'\d+', city) or '@' in city:
        return np.nan

    # 2. Eliminar acentos para coherencia con el resto del dataset
    city = unicodedata.normalize('NFKD', city)
    city = ''.join(c for c in city if not unicodedata.combining(c))

    # 3. Normalizar apóstrofes especiales
    city = city.replace('´', "'")

    # 4. Extraer parte antes de / o \ (formatos ciudad/estado o ciudad\estado)
    city = re.split(r'[/\\]', city)[0].strip()

    # 5. Extraer parte antes de la primera coma (formatos ciudad, estado, país)
    city = city.split(',')[0].strip()

    # 6. Eliminar contenido entre paréntesis
    city = re.sub(r'\s*\(.*?\)', '', city).strip()

    # 7. Correcciones puntuales
    city = REPLACEMENTS.get(city, city)

    return city


# Guardar valores originales para el reporte
df['seller_city_original'] = df['seller_city']

# Aplicar limpieza
df['seller_city'] = df['seller_city'].apply(clean_city)

print("Limpieza aplicada. Resumen de cambios:")
mascara_cambio = df['seller_city'] != df['seller_city_original']
print(f"  Filas modificadas : {mascara_cambio.sum()}")
print(f"  Nuevos NaN        : {df['seller_city'].isnull().sum()}")


In [ ]:
# Mostrar las transformaciones realizadas
cambios = df[mascara_cambio][['seller_city_original', 'seller_city', 'seller_state']].drop_duplicates()
cambios.columns = ['ciudad_original', 'ciudad_limpia', 'estado']
cambios.sort_values('ciudad_original').reset_index(drop=True)


## 4. Verificación de `seller_zip_code_prefix`

El código postal fue cargado como `str` para preservar ceros iniciales. Se verifica que todos los valores tengan exactamente 5 dígitos y se aplica zero-padding donde sea necesario.

In [ ]:
zips = df['seller_zip_code_prefix']

no_numericos = zips[~zips.str.fullmatch(r'\d+')].unique().tolist()
longitud_incorrecta = zips[zips.str.len() != 5].unique().tolist()

print(f"Zip codes no numéricos      : {len(no_numericos)}")
print(f"Zip codes con longitud ≠ 5  : {len(longitud_incorrecta)}")
print(f"Con cero inicial            : {(zips.str[0] == '0').sum()}")

# Aplicar zero-padding por si algún valor tuviese menos de 5 dígitos
df['seller_zip_code_prefix'] = zips.str.zfill(5)

print("\nEjemplos de zip_code_prefix tras normalización:")
print(df['seller_zip_code_prefix'].head(10).tolist())


## 5. Reporte final

In [ ]:
ciudades_antes = df['seller_city_original'].nunique()
df_clean = df.drop(columns=['seller_city_original'])

print("=" * 50)
print("REPORTE DE LIMPIEZA")
print("=" * 50)
print(f"\nFilas totales            : {len(df_clean)}")
print(f"Columnas                 : {list(df_clean.columns)}")
print(f"\nValores nulos tras limpieza:")
print('Nulos por columna en df_clean:')
print(df_clean.isnull().sum().to_string())
print(f"\nFilas con seller_city nulo (irrecuperables): {df_clean['seller_city'].isnull().sum()}")
print(f"Ciudades únicas (antes)  : {ciudades_antes}")
print(f"Ciudades únicas (después): {df_clean['seller_city'].nunique()}")
print(f"\nEstados únicos           : {sorted(df_clean['seller_state'].unique().tolist())}")
print(f"\nTop 10 ciudades (limpias):")
print(df_clean['seller_city'].value_counts().head(10).to_string())


## 6. Exportar dataset limpio

In [ ]:
# Guardar en la carpeta de salida configurada
output_path = guardar_csv_limpio(
    df_clean,
    'olist_sellers_dataset_clean.csv',
    PIPELINE_AUDIT['leidos']['olist_sellers_dataset.csv']['filas']
)
print(f"Dataset limpio guardado en: {os.path.abspath(output_path)}")
print(f"Forma final: {df_clean.shape}")

## Validacion local de salidas del script 01


In [ ]:
archivos_esperados = [
    'olist_customers_clean.csv',
    'olist_order_items_clean.csv',
    'olist_geolocation_clean.csv',
    'olist_products_clean.csv',
    'olist_orders_clean.csv',
    'olist_order_payments_dataset_clean.csv',
    'product_category_name_translation_clean.csv',
    'olist_order_reviews_dataset_clean.csv',
    'olist_sellers_dataset_clean.csv',
]

print('=' * 80)
print('VALIDACION LOCAL - SCRIPT 01')
print('=' * 80)
print(f'Carpeta de entrada: {os.path.abspath(DATA_PATH)}')
print(f'Carpeta de salida : {os.path.abspath(OUTPUT_PATH)}')

print('\nArchivos leidos:')
for nombre, info in PIPELINE_AUDIT['leidos'].items():
    print(f"- {nombre}: {info['filas']:,} filas | {len(info['columnas'])} columnas | {info['ruta']}")

print('\nArchivos generados:')
for nombre, info in PIPELINE_AUDIT['generados'].items():
    print(f"- {nombre}: {info['filas_antes']:,} -> {info['filas_despues']:,} filas | {len(info['columnas_finales'])} columnas | {info['ruta']}")
    print(f"  Columnas finales: {info['columnas_finales']}")
    print(f"  Valores nulos principales: {info['nulos_principales']}")

faltantes = [nombre for nombre in archivos_esperados if not os.path.exists(output_csv_path(nombre))]
no_registrados = [nombre for nombre in archivos_esperados if nombre not in PIPELINE_AUDIT['generados']]

if faltantes or no_registrados:
    if faltantes:
        print('\nArchivos esperados no encontrados en salida:')
        for nombre in faltantes:
            print(f'- {nombre}')
    if no_registrados:
        print('\nArchivos esperados sin registro de auditoria:')
        for nombre in no_registrados:
            print(f'- {nombre}')
    raise FileNotFoundError('La validacion local detecto salidas faltantes.')

print('\nOK: todos los CSV limpios esperados se generaron correctamente en csv/csv_limpios/.')